In [1]:
import torch
import pandas as pd
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, EsmForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import (
    confusion_matrix, accuracy_score, roc_auc_score, matthews_corrcoef,
    precision_score, recall_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight

# "facebook/esm2_t6_8M_UR50D"  (Lightweight, fast)
# "facebook/esm2_t12_35M_UR50D" (Medium)
#esm2_t30_150M_UR50D (heavy, good)
# "facebook/esm2_t33_650M_UR50D" (Heavy, requires good GPU)
MODEL_NAME = "facebook/esm2_t6_8M_UR50D" 
BATCH_SIZE = 16
LEARNING_RATE = 1e-6
EPOCHS = 3
MAX_LEN = 250   

if torch.cuda.is_available():
    device = torch.device("cuda")
   
else:
    device = torch.device("cpu")

/home/protein/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_data(file_path):
    try:
        df = pd.read_csv(file_path)
        return df
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        raise

train_data = load_data("PI_nonPI_train_80.csv")
test_data = load_data("PI_nonPI_test_20.csv")

train_texts = train_data['sequence'].tolist()
train_labels = train_data['label'].tolist()
test_texts = test_data['sequence'].tolist()
test_labels = test_data['label'].tolist()

print(f"Training samples: {len(train_texts)}")
print(f"Testing samples: {len(test_texts)}")

class_weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(train_labels), 
    y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights computed: {class_weights}")

Training samples: 18101
Testing samples: 4526
Class Weights computed: [0.73159001 1.57949389]


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prepare_tensors(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
    labels_tensor = torch.tensor(labels)
    return TensorDataset(encodings['input_ids'], encodings['attention_mask'], labels_tensor)

train_dataset = prepare_tensors(train_texts, train_labels)
test_dataset = prepare_tensors(test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [4]:
model = EsmForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps=0, 
                                            num_training_steps=total_steps)

criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)


Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
start_time = time.time()

training_stats = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for step, batch in enumerate(train_loader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        model.zero_grad()

        outputs = model(b_input_ids, attention_mask=b_input_mask)
        logits = outputs.logits

        loss = criterion(logits, b_labels)
        total_loss += loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        
    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Average Loss: {avg_train_loss:.4f}")
    training_stats.append({'epoch': epoch + 1, 'loss': avg_train_loss})

print(f" {time.time() - start_time:.2f} seconds.")

Epoch 1/3 | Average Loss: 0.5852
Epoch 2/3 | Average Loss: 0.3882
Epoch 3/3 | Average Loss: 0.3271
 297.57 seconds.


In [6]:
model.eval()

all_preds = []
all_probs = []
all_labels_actual = []

with torch.no_grad():
    for batch in test_loader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        outputs = model(b_input_ids, attention_mask=b_input_mask)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        probs = torch.softmax(logits, dim=1).cpu().numpy()[:, 1] 
        
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_labels_actual.extend(b_labels.cpu().numpy())

y_true = all_labels_actual
y_pred = all_preds
y_prob = all_probs

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1 Score:    {f1:.4f}")
print(f"ROC-AUC:     {roc_auc:.4f}")
print(f"MCC:         {mcc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")


Accuracy:    0.9317
Precision:   0.8903
Recall:      0.8946
F1 Score:    0.8924
ROC-AUC:     0.9715
MCC:         0.8424
Sensitivity: 0.8946
Specificity: 0.9489


In [7]:
import os

MODEL_SAVE_PATH = "esm_binary_classification_model_150M"

if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)

print(f"Saving model to {MODEL_SAVE_PATH}...")
model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)


Saving model to esm_binary_classification_model_150M...


('esm_binary_classification_model_150M/tokenizer_config.json',
 'esm_binary_classification_model_150M/special_tokens_map.json',
 'esm_binary_classification_model_150M/vocab.txt',
 'esm_binary_classification_model_150M/added_tokens.json')

In [8]:
import torch
import pandas as pd
from transformers import AutoTokenizer, EsmForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os

INPUT_FILE = "MAPH_permissive_noPfam_250AA_sequences.csv"  
OUTPUT_FILE = "ESM_150M_inference_results.csv"
MODEL_PATH = "esm_binary_classification_model_150M" 
MAX_LEN = 250
BATCH_SIZE = 16

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Could not find the file: {INPUT_FILE}")

df_inference = pd.read_csv(INPUT_FILE)

if 'header' not in df_inference.columns or 'sequence' not in df_inference.columns:
    raise ValueError(f"CSV must contain 'header' and 'sequence' columns. Found: {df_inference.columns}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model folder '{MODEL_PATH}' not found. Did you run the 'Save Model' block?")

print(f"Loading model from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = EsmForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(device)
model.eval() 

sequences = df_inference['sequence'].astype(str).tolist()

encoded_inputs = tokenizer(
    sequences, 
    padding=True, 
    truncation=True, 
    max_length=MAX_LEN, 
    return_tensors="pt"
)

inference_dataset = TensorDataset(encoded_inputs.input_ids, encoded_inputs.attention_mask)
inference_dataloader = DataLoader(inference_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_probs = []
all_preds = []

with torch.no_grad():
    for batch in inference_dataloader:
        input_ids, attention_mask = [b.to(device) for b in batch]

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1) 
        
        all_probs.extend(probs)
        all_preds.extend(preds)

confidence_scores = [p[pred] for p, pred in zip(all_probs, all_preds)]

df_inference['predicted_class_id'] = all_preds 
df_inference['confidence'] = confidence_scores
df_inference['prob_class_1'] = [p[1] for p in all_probs] 

df_inference.to_csv(OUTPUT_FILE, index=False)

print(df_inference[['header', 'predicted_class_id', 'confidence']].head())

counts = df_inference['predicted_class_id'].value_counts()
total = len(df_inference)

for label, count in counts.items():
    print(f"Class {label}: {count} ({count/total*100:.1f}%)")

Using device: cuda
Loading model from esm_binary_classification_model_150M...
   header  predicted_class_id  confidence
0  K2QGL0                   0    0.515073
1  K2QIM2                   0    0.733948
2  K2QIQ7                   1    0.806825
3  K2QIZ1                   0    0.592010
4  K2QIZ9                   1    0.826529
Class 1: 282 (64.5%)
Class 0: 155 (35.5%)
